In [ ]:
# Run once per Colab session
!pip -q install tensorflow-datasets tensorflow scikit-learn matplotlib seaborn


In [ ]:
import os, time, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, initializers
import tensorflow_datasets as tfds

from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Use GPU if Colab gives you one (Runtime > Change runtime type > GPU)
print("GPUs:", tf.config.list_physical_devices('GPU'))

IMG_SIZE = 224
NUM_CLASSES = 37          # 37 pet breeds
EPOCHS_SMALL = 5          # fast exploratory runs (initialization/regularization/optimizer sweeps)
EPOCHS_FULL  = 15         # final / fine-tuning runs
BATCH_SIZE   = 32
AUTOTUNE = tf.data.AUTOTUNE


In [ ]:
(ds_train_full, ds_test), ds_info = tfds.load(
    'oxford_iiit_pet',
    split=['train', 'test'],
    with_info=True,
    as_supervised=False,   # keep dict so we can pull 'image' and 'label'
)

print(ds_info.features)
n_train_full = ds_info.splits['train'].num_examples
n_test = ds_info.splits['test'].num_examples
print("train_full:", n_train_full, "  test:", n_test)


In [ ]:
def preprocess(example, augment=False):
    image = tf.cast(example['image'], tf.float32)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    if augment:
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, 0.1)
    # MobileNetV2 preprocessing: scales pixels to [-1, 1]
    image = keras.applications.mobilenet_v2.preprocess_input(image)
    label = tf.cast(example['label'], tf.int32)
    return image, label

def make_split(ds, take=None, skip=None, batch_size=BATCH_SIZE, shuffle=False, augment=False):
    if skip is not None:
        ds = ds.skip(skip)
    if take is not None:
        ds = ds.take(take)
    if shuffle:
        ds = ds.shuffle(1000, seed=SEED)
    ds = ds.map(lambda e: preprocess(e, augment=augment), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

# 85/15 train/validation split out of the given training data
n_val = int(0.15 * n_train_full)
n_train = n_train_full - n_val

train_ds = make_split(ds_train_full, skip=n_val, shuffle=True, augment=True)
val_ds   = make_split(ds_train_full, take=n_val, shuffle=False, augment=False)
test_ds  = make_split(ds_test, shuffle=False, augment=False)

print(f"n_train={n_train}  n_val={n_val}  n_test={n_test}")

# Peek at a batch
for imgs, labels in train_ds.take(1):
    print(imgs.shape, labels.shape)


In [ ]:
def build_model(weight_init='he', dropout_rate=0.5, l2_reg=0.0, use_bn=True,
                 pretrained=True, freeze_base=True, unfreeze_from=None,
                 learning_rate=1e-3, optimizer_name='adam'):
    """Builds a MobileNetV2-based classifier for the 37-class pet dataset."""

    init_map = {
        'zeros': initializers.Zeros(),
        'random': initializers.RandomNormal(mean=0.0, stddev=0.05, seed=SEED),
        'glorot': initializers.GlorotUniform(seed=SEED),
        'he': initializers.HeNormal(seed=SEED),
    }
    kernel_init = init_map[weight_init]
    reg = keras.regularizers.l2(l2_reg) if l2_reg > 0 else None

    base = keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet' if pretrained else None,
        pooling='avg',
    )

    if pretrained:
        base.trainable = not freeze_base
        if not freeze_base and unfreeze_from is not None:
            # freeze everything before `unfreeze_from`, train the rest (fine-tuning)
            for layer in base.layers[:unfreeze_from]:
                layer.trainable = False
            for layer in base.layers[unfreeze_from:]:
                layer.trainable = True
    else:
        base.trainable = True  # training from scratch, to study init/reg/BN in isolation

    x = base.output
    if use_bn:
        x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(128, activation='relu', kernel_initializer=kernel_init,
                      kernel_regularizer=reg)(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax',
                            kernel_initializer=kernel_init, kernel_regularizer=reg)(x)

    model = keras.Model(base.input, outputs)

    opt_map = {
        'sgd': optimizers.SGD(learning_rate=learning_rate),
        'momentum': optimizers.SGD(learning_rate=learning_rate, momentum=0.9),
        'rmsprop': optimizers.RMSprop(learning_rate=learning_rate),
        'adam': optimizers.Adam(learning_rate=learning_rate),
    }
    model.compile(optimizer=opt_map[optimizer_name],
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


def train_and_time(model, train_ds, val_ds, epochs, verbose=0):
    t0 = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=verbose)
    elapsed = time.time() - t0
    return history, elapsed


In [ ]:
init_histories = {}
for init_name in ['zeros', 'random', 'glorot', 'he']:
    print(f"--- Training with {init_name} initialization ---")
    m = build_model(weight_init=init_name, pretrained=False, dropout_rate=0.3,
                     optimizer_name='adam', learning_rate=1e-3)
    hist, _ = train_and_time(m, train_ds, val_ds, epochs=EPOCHS_SMALL, verbose=1)
    init_histories[init_name] = hist.history


In [ ]:
# Plot 1: Training Loss vs Epoch
plt.figure(figsize=(7,5))
for name, h in init_histories.items():
    plt.plot(h['loss'], label=name)
plt.xlabel('Epoch'); plt.ylabel('Training Loss')
plt.title('Plot 1: Training Loss vs Epoch (Weight Initialization)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

# Plot 2: Validation Accuracy vs Epoch
plt.figure(figsize=(7,5))
for name, h in init_histories.items():
    plt.plot([a*100 for a in h['val_accuracy']], label=name)
plt.xlabel('Epoch'); plt.ylabel('Validation Accuracy (%)')
plt.title('Plot 2: Validation Accuracy vs Epoch (Weight Initialization)')
plt.legend(); plt.grid(alpha=0.3); plt.show()


In [ ]:
reg_configs = {
    'no_reg':   dict(dropout_rate=0.0, l2_reg=0.0,   use_bn=False),
    'l2':       dict(dropout_rate=0.0, l2_reg=1e-3,  use_bn=False),
    'dropout':  dict(dropout_rate=0.5, l2_reg=0.0,   use_bn=False),
    'batchnorm':dict(dropout_rate=0.0, l2_reg=0.0,   use_bn=True),
}

reg_histories = {}
for name, cfg in reg_configs.items():
    print(f"--- Training with {name} ---")
    m = build_model(weight_init='he', pretrained=True, freeze_base=True,
                     optimizer_name='adam', learning_rate=1e-3, **cfg)
    hist, _ = train_and_time(m, train_ds, val_ds, epochs=EPOCHS_SMALL, verbose=1)
    reg_histories[name] = hist.history


In [ ]:
# Plot 3: Training and Validation Accuracy vs Epoch (per configuration)
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (name, h) in zip(axes.flat, reg_histories.items()):
    ax.plot([a*100 for a in h['accuracy']], label='train')
    ax.plot([a*100 for a in h['val_accuracy']], label='val')
    ax.set_title(name); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Plot 3: Training vs Validation Accuracy by Regularization Scheme')
plt.tight_layout(); plt.show()

# Plot 4: Training and Validation Loss vs Epoch
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (name, h) in zip(axes.flat, reg_histories.items()):
    ax.plot(h['loss'], label='train')
    ax.plot(h['val_loss'], label='val')
    ax.set_title(name); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Plot 4: Training vs Validation Loss by Regularization Scheme')
plt.tight_layout(); plt.show()


In [ ]:
x = np.array([2., 4., 6., 8.])
mu = x.mean()
var = x.var()  # population variance, matches 1/m formula
x_hat = (x - mu) / np.sqrt(var + 1e-8)
print("mean:", mu, " variance:", var, " std:", np.sqrt(var))
print("x_hat:", np.round(x_hat, 3))
gamma, beta = 1.0, 0.0
y = gamma * x_hat + beta
print("y (gamma=1, beta=0):", np.round(y, 3))


In [ ]:
# Plot 5: With vs Without Batch Normalization (val accuracy)
plt.figure(figsize=(7,5))
plt.plot([a*100 for a in reg_histories['batchnorm']['val_accuracy']], label='With BN')
plt.plot([a*100 for a in reg_histories['no_reg']['val_accuracy']], label='Without BN')
plt.xlabel('Epoch'); plt.ylabel('Validation Accuracy (%)')
plt.title('Plot 5: With vs Without Batch Normalization')
plt.legend(); plt.grid(alpha=0.3); plt.show()


In [ ]:
opt_histories = {}
opt_times = {}
for opt_name in ['sgd', 'momentum', 'rmsprop', 'adam']:
    print(f"--- Training with optimizer={opt_name} ---")
    m = build_model(weight_init='he', pretrained=True, freeze_base=True,
                     dropout_rate=0.3, optimizer_name=opt_name, learning_rate=1e-3)
    hist, elapsed = train_and_time(m, train_ds, val_ds, epochs=EPOCHS_SMALL, verbose=1)
    opt_histories[opt_name] = hist.history
    opt_times[opt_name] = elapsed


In [ ]:
# Plot 6: Training Loss vs Epoch (optimizers)
plt.figure(figsize=(7,5))
for name, h in opt_histories.items():
    plt.plot(h['loss'], label=name)
plt.xlabel('Epoch'); plt.ylabel('Training Loss')
plt.title('Plot 6: Training Loss vs Epoch by Optimizer')
plt.legend(); plt.grid(alpha=0.3); plt.show()

# Plot 7: Validation Accuracy vs Epoch (optimizers)
plt.figure(figsize=(7,5))
for name, h in opt_histories.items():
    plt.plot([a*100 for a in h['val_accuracy']], label=name)
plt.xlabel('Epoch'); plt.ylabel('Validation Accuracy (%)')
plt.title('Plot 7: Validation Accuracy vs Epoch by Optimizer')
plt.legend(); plt.grid(alpha=0.3); plt.show()


In [ ]:
# Optimizer summary table
rows = []
for name, h in opt_histories.items():
    best_val = max(h['val_accuracy']) * 100
    epoch_to_converge = int(np.argmax(h['val_accuracy'])) + 1
    rows.append({
        'Optimizer': name,
        'Final Loss': round(h['loss'][-1], 4),
        'Best Val. Accuracy (%)': round(best_val, 2),
        'Epoch to Converge': epoch_to_converge,
        'Time (s)': round(opt_times[name], 1),
    })
opt_table = pd.DataFrame(rows)
opt_table


In [ ]:
def rebatch(ds_raw, take=None, skip=None, batch_size=BATCH_SIZE, shuffle=False, augment=False):
    return make_split(ds_raw, take=take, skip=skip, batch_size=batch_size,
                       shuffle=shuffle, augment=augment)

# --- Learning rate sweep ---
lr_results = {}
for lr in [1e-3, 1e-4]:
    m = build_model(weight_init='he', pretrained=True, freeze_base=True,
                     dropout_rate=0.3, optimizer_name='adam', learning_rate=lr)
    hist, _ = train_and_time(m, train_ds, val_ds, epochs=EPOCHS_SMALL, verbose=0)
    lr_results[lr] = max(hist.history['val_accuracy']) * 100
    print(f"lr={lr}: best val acc = {lr_results[lr]:.2f}%")

# --- Batch size sweep ---
bs_results = {}
for bs in [16, 32, 64]:
    tr = rebatch(ds_train_full, skip=n_val, batch_size=bs, shuffle=True, augment=True)
    va = rebatch(ds_train_full, take=n_val, batch_size=bs, shuffle=False, augment=False)
    m = build_model(weight_init='he', pretrained=True, freeze_base=True,
                     dropout_rate=0.3, optimizer_name='adam', learning_rate=1e-3)
    hist, _ = train_and_time(m, tr, va, epochs=EPOCHS_SMALL, verbose=0)
    bs_results[bs] = max(hist.history['val_accuracy']) * 100
    print(f"batch_size={bs}: best val acc = {bs_results[bs]:.2f}%")

# --- Dropout rate sweep ---
do_results = {}
for do in [0.0, 0.25, 0.5]:
    m = build_model(weight_init='he', pretrained=True, freeze_base=True,
                     dropout_rate=do, optimizer_name='adam', learning_rate=1e-3)
    hist, _ = train_and_time(m, train_ds, val_ds, epochs=EPOCHS_SMALL, verbose=0)
    do_results[do] = max(hist.history['val_accuracy']) * 100
    print(f"dropout={do}: best val acc = {do_results[do]:.2f}%")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16,4.5))

axes[0].plot(list(map(str, lr_results.keys())), list(lr_results.values()), marker='o')
axes[0].set_title('Plot 8: Learning Rate vs Val. Accuracy')
axes[0].set_xlabel('Learning Rate'); axes[0].set_ylabel('Validation Accuracy (%)')

axes[1].plot(list(map(str, bs_results.keys())), list(bs_results.values()), marker='o')
axes[1].set_title('Plot 9: Batch Size vs Val. Accuracy')
axes[1].set_xlabel('Batch Size'); axes[1].set_ylabel('Validation Accuracy (%)')

axes[2].plot(list(map(str, do_results.keys())), list(do_results.values()), marker='o')
axes[2].set_title('Plot 10: Dropout Rate vs Val. Accuracy')
axes[2].set_xlabel('Dropout Rate'); axes[2].set_ylabel('Validation Accuracy (%)')

for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# Case A: Feature extraction
model_fe = build_model(weight_init='he', pretrained=True, freeze_base=True,
                        dropout_rate=0.3, optimizer_name='adam', learning_rate=1e-3)
hist_fe, _ = train_and_time(model_fe, train_ds, val_ds, epochs=EPOCHS_SMALL, verbose=1)

# Case B: Fine-tuning — start from the feature-extraction model, unfreeze the last ~30 layers
base_layer_count = len(model_fe.layers)
model_ft = build_model(weight_init='he', pretrained=True, freeze_base=False,
                        unfreeze_from=base_layer_count - 30,
                        dropout_rate=0.3, optimizer_name='adam', learning_rate=1e-5)
model_ft.set_weights(model_fe.get_weights())  # continue from the trained head
hist_ft, _ = train_and_time(model_ft, train_ds, val_ds, epochs=EPOCHS_SMALL, verbose=1)


In [ ]:
# Plot 11: Feature Extraction vs Fine-Tuning
plt.figure(figsize=(7,5))
plt.plot([a*100 for a in hist_fe.history['val_accuracy']], label='Feature Extraction')
plt.plot([a*100 for a in hist_ft.history['val_accuracy']], label='Fine-Tuning')
plt.xlabel('Epoch'); plt.ylabel('Validation Accuracy (%)')
plt.title('Plot 11: Feature Extraction vs Fine-Tuning')
plt.legend(); plt.grid(alpha=0.3); plt.show()

# Plot 12: Loss before/after fine-tuning
plt.figure(figsize=(7,5))
plt.plot(hist_fe.history['loss'], label='Train loss (Feature Extraction)')
plt.plot(hist_fe.history['val_loss'], label='Val loss (Feature Extraction)')
plt.plot(hist_ft.history['loss'], '--', label='Train loss (Fine-Tuning)')
plt.plot(hist_ft.history['val_loss'], '--', label='Val loss (Fine-Tuning)')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Plot 12: Training & Validation Loss — Before vs After Fine-Tuning')
plt.legend(); plt.grid(alpha=0.3); plt.show()


In [ ]:
# Materialize the full training pool (train+val together) as numpy arrays for KFold indexing.
# For speed we cap the number of samples used for the CV demo (raise SUBSET or set to None for the full set).
SUBSET = 1500

def dataset_to_arrays(ds_raw, limit=None):
    imgs, labels = [], []
    ds = ds_raw
    if limit is not None:
        ds = ds.take(limit)
    for ex in tfds.as_numpy(ds):
        img = tf.image.resize(ex['image'], (IMG_SIZE, IMG_SIZE)).numpy()
        img = keras.applications.mobilenet_v2.preprocess_input(img)
        imgs.append(img)
        labels.append(ex['label'])
    return np.array(imgs, dtype=np.float32), np.array(labels, dtype=np.int32)

X_pool, y_pool = dataset_to_arrays(ds_train_full, limit=SUBSET)
print(X_pool.shape, y_pool.shape)


In [ ]:
cv_configs = {
    'C1_baseline':        dict(weight_init='he', dropout_rate=0.3, l2_reg=0.0,  use_bn=False, optimizer_name='adam',    learning_rate=1e-3),
    'C2_dropout_bn':      dict(weight_init='he', dropout_rate=0.5, l2_reg=0.0,  use_bn=True,  optimizer_name='adam',    learning_rate=1e-3),
    'C3_l2':              dict(weight_init='he', dropout_rate=0.3, l2_reg=1e-3, use_bn=False, optimizer_name='adam',    learning_rate=1e-4),
    'C4_rmsprop':         dict(weight_init='he', dropout_rate=0.3, l2_reg=0.0,  use_bn=False, optimizer_name='rmsprop', learning_rate=1e-3),
}

K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=SEED)
cv_results = {name: [] for name in cv_configs}

for name, cfg in cv_configs.items():
    print(f"=== Config {name} ===")
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_pool)):
        Xtr, ytr = X_pool[train_idx], y_pool[train_idx]
        Xva, yva = X_pool[val_idx], y_pool[val_idx]
        m = build_model(pretrained=True, freeze_base=True, **cfg)
        m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCHS_SMALL,
              batch_size=BATCH_SIZE, verbose=0)
        val_acc = m.evaluate(Xva, yva, verbose=0)[1] * 100
        cv_results[name].append(val_acc)
        print(f"  fold {fold+1}: val acc = {val_acc:.2f}%")
        keras.backend.clear_session()


In [ ]:
cv_rows = []
for name, accs in cv_results.items():
    row = {f'F{i+1}': round(a, 2) for i, a in enumerate(accs)}
    row['Mean ± SD'] = f"{np.mean(accs):.2f} ± {np.std(accs):.2f}"
    row['Configuration'] = name
    cv_rows.append(row)
cv_table = pd.DataFrame(cv_rows).set_index('Configuration')
cv_table = cv_table[['F1','F2','F3','F4','F5','Mean ± SD']]
cv_table


In [ ]:
# Plot 13: 5-Fold CV Accuracy with error bars
means = [np.mean(v) for v in cv_results.values()]
stds  = [np.std(v) for v in cv_results.values()]
names = list(cv_results.keys())

plt.figure(figsize=(8,5))
plt.bar(names, means, yerr=stds, capsize=6)
plt.xlabel('Hyperparameter Configuration'); plt.ylabel('Mean Validation Accuracy (%)')
plt.title('Plot 13: 5-Fold Cross-Validation Accuracy (± SD)')
plt.grid(alpha=0.3, axis='y'); plt.show()

best_config_name = names[int(np.argmax(means))]
print("Best configuration by mean CV accuracy:", best_config_name)


In [ ]:
best_cfg = cv_configs[best_config_name]

final_model = build_model(pretrained=True, freeze_base=True, **best_cfg)

t0 = time.time()
final_hist = final_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FULL, verbose=1)
final_train_time = time.time() - t0

test_loss, test_acc = final_model.evaluate(test_ds, verbose=1)
print(f"Test accuracy: {test_acc*100:.2f}%")


In [ ]:
# Predictions on the test set for precision/recall/F1/confusion matrix
y_true, y_pred = [], []
for imgs, labels in test_ds:
    preds = final_model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())
y_true = np.array(y_true); y_pred = np.array(y_pred)

precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall    = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1        = f1_score(y_true, y_pred, average='macro', zero_division=0)
n_params  = final_model.count_params()

final_metrics = pd.DataFrame({
    'Metric': ['Mean CV Accuracy', 'CV Standard Deviation', 'Test Accuracy',
               'Precision', 'Recall', 'F1-score', 'Training Time (s)', 'Number of Parameters'],
    'Value': [
        f"{np.mean(cv_results[best_config_name]):.2f}%",
        f"{np.std(cv_results[best_config_name]):.2f}",
        f"{test_acc*100:.2f}%",
        round(precision, 3), round(recall, 3), round(f1, 3),
        round(final_train_time, 1), n_params,
    ]
})
final_metrics


In [ ]:
# Plot 14: Confusion Matrix
class_names = ds_info.features['label'].names
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14,12))
sns.heatmap(cm, cmap='viridis', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Plot 14: Confusion Matrix — Final Model on Test Set')
plt.xticks(rotation=90); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

# Most confused class pairs
cm_offdiag = cm.copy()
np.fill_diagonal(cm_offdiag, 0)
top_confusions = np.dstack(np.unravel_index(np.argsort(-cm_offdiag.ravel())[:5], cm.shape))[0]
print("Most frequently confused class pairs (true -> predicted, count):")
for i, j in top_confusions:
    print(f"  {class_names[i]} -> {class_names[j]}: {cm[i, j]}")


In [ ]:
# Optional Plot 15: Misclassified images
misclassified_idx = np.where(y_true != y_pred)[0]
sample_idx = np.random.choice(misclassified_idx, size=min(6, len(misclassified_idx)), replace=False)

# Re-fetch raw (unnormalized) images for display
raw_test_imgs = []
for ex in tfds.as_numpy(ds_test.take(200)):
    raw_test_imgs.append(tf.image.resize(ex['image'], (IMG_SIZE, IMG_SIZE)).numpy().astype('uint8'))

fig, axes = plt.subplots(1, len(sample_idx), figsize=(4*len(sample_idx), 4))
for ax, idx in zip(np.atleast_1d(axes), sample_idx):
    if idx < len(raw_test_imgs):
        ax.imshow(raw_test_imgs[idx])
    ax.set_title(f"True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}", fontsize=9)
    ax.axis('off')
plt.suptitle('Plot 15 (Optional): Misclassified Images')
plt.tight_layout(); plt.show()


In [ ]:
overall_results = pd.DataFrame({
    'Configuration': ['Baseline', 'Best Initialization', 'Best Regularization',
                       'Best Optimizer', 'Best Hyperparameters', 'Fine-Tuned Model'],
    'CV Accuracy (%)': [None]*6,
    'SD': [None]*6,
    'Test Accuracy (%)': [None]*6,
    'Training Time (s)': [None]*6,
})
# Populate this by hand from the sweeps above (e.g. overall_results.loc[0, 'CV Accuracy (%)'] = ...)
overall_results
